In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS olist.gold")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_fat_pedido_total  = spark.table("olist.silver.fat_pedido_total")
df_fat_itens_pedidos = spark.table("olist.silver.fat_itens_pedidos")
df_dim_produtos      = spark.table("olist.silver.dim_produtos")
df_dim_categoria     = spark.table("olist.silver.dim_categoria_produtos_traducao")

In [0]:
# Enriquece itens com categoria do produto e tradução para inglês
df_itens_enriquecido = (
    df_fat_itens_pedidos
    .join(df_dim_produtos, "id_produto", "left")
    .join(
        df_dim_categoria,
        df_dim_produtos["categoria_produto"] == df_dim_categoria["nome_produto_pt"],
        "left"
    )
    .select(
        "id_pedido",
        "id_produto",
        "preco_BRL",
        F.coalesce(
            F.col("nome_produto_en"),
            F.col("categoria_produto")
        ).alias("categoria_produto")
    )
)

# Junta com fat_pedido_total para obter data e cotação do dólar
df_vendas = (
    df_fat_pedido_total
    .join(df_itens_enriquecido, "id_pedido", "inner")
    .withColumn("ano_venda", F.year("data_pedido"))
    .withColumn("mes_venda", F.month("data_pedido"))
    # Calcula valor em USD por item usando a cotação já presente no fat_pedido_total
    .withColumn(
        "preco_usd",
        F.when(
            F.col("valor_total_pago_brl") > 0,
            F.col("preco_BRL") * (F.col("valor_total_pago_usd") / F.col("valor_total_pago_brl"))
        ).otherwise(F.lit(0))
    )
)

# Agrega por Ano, Mês e Categoria
df_fat_vendas_comercial = (
    df_vendas
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("id_produto").alias("qtd_itens_vendidos"),
        F.round(F.sum("preco_BRL"), 2).alias("receita_total_brl"),
        F.round(F.sum("preco_usd"), 2).alias("receita_total_usd"),
        F.round(F.sum("preco_BRL") / F.countDistinct("id_pedido"), 2).alias("ticket_medio_brl"),
    )
    .orderBy("ano_venda", "mes_venda")
)

(
    df_fat_vendas_comercial.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.gold.fat_vendas_comercial")
)

print(f"✅ olist.gold.fat_vendas_comercial ({df_fat_vendas_comercial.count()} registros)")

In [0]:
df_produtos_raw = spark.table("olist.bronze.tb_products").select(
    F.col("product_id").alias("id_produto"),
    F.col("product_name").alias("nome_produto"),
    F.col("product_category_name").alias("categoria_produto_pt")
)

df_produtos_enriquecido = (
    df_produtos_raw
    .join(df_dim_categoria, df_produtos_raw["categoria_produto_pt"] == df_dim_categoria["nome_produto_pt"], "left")
    .select(
        "id_produto",
        "nome_produto",
        F.coalesce(df_dim_categoria["nome_produto_en"], F.col("categoria_produto_pt")).alias("categoria_produto")
    )
)

df_ranking_produtos = (
    df_fat_itens_pedidos
    .join(df_produtos_enriquecido, "id_produto", "left")
    .groupBy("id_produto", "nome_produto", "categoria_produto")
    .agg(F.count("id_produto").alias("quantidade_vendida"))
)

df_top5_mais = (
    df_ranking_produtos
    .orderBy(F.col("quantidade_vendida").desc())
    .limit(5)
)
print("🏆 Top 5 Produtos MAIS Vendidos:")
display(df_top5_mais)

df_top5_menos = (
    df_ranking_produtos
    .orderBy(F.col("quantidade_vendida").asc())
    .limit(5)
)
print("📉 Top 5 Produtos MENOS Vendidos:")
display(df_top5_menos)

In [0]:
df_fat_avaliacoes = spark.table("olist.silver.fat_avaliacoes_pedidos")
df_fat_itens      = spark.table("olist.silver.fat_itens_pedidos")
df_dim_vendedores = spark.table("olist.silver.dim_vendedores")

In [0]:
df_produtos_aval = spark.table("olist.bronze.tb_products").select(
    F.col("product_id").alias("id_produto"),
    F.col("product_name").alias("nome_produto"),
    F.col("product_category_name").alias("categoria_produto_pt")
)

df_dim_categoria_aval = spark.table("olist.silver.dim_categoria_produtos_traducao")

# Enriquece produtos com categoria traduzida
df_produtos_cat_aval = (
    df_produtos_aval
    .join(
        df_dim_categoria_aval,
        df_produtos_aval["categoria_produto_pt"] == df_dim_categoria_aval["nome_produto_pt"],
        "left"
    )
    .select(
        "id_produto",
        "nome_produto",
        F.coalesce(
            df_dim_categoria_aval["nome_produto_en"],
            F.col("categoria_produto_pt")
        ).alias("categoria_produto")
    )
)

df_avaliacoes_completo = (
    df_fat_avaliacoes
    .join(df_fat_itens, "id_pedido", "left")
    .join(df_produtos_cat_aval, "id_produto", "left")
    .join(df_dim_vendedores, "id_vendedor", "left")
    .select(
        "id_pedido",
        "nota_avaliacao",
        "nome_produto",
        "categoria_produto",
        df_dim_vendedores["id_vendedor"].alias("id_vendedor"),
        df_dim_vendedores["estado"].alias("estado_vendedor"),
    )
)

In [0]:
df_avaliacoes_completo = df_avaliacoes_completo.withColumn(
    "nota_avaliacao",
    F.expr("try_cast(nota_avaliacao as double)")
).filter(F.col("nota_avaliacao").isNotNull())

df_fat_avaliacoes_clientes = (
    df_avaliacoes_completo
    .groupBy("categoria_produto", "id_vendedor", "estado_vendedor")
    .agg(
        F.count("nota_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas"),
        F.round(
            F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)) /
            F.count("nota_avaliacao") * 100, 2
        ).alias("percentual_satisfacao"),
    )
)

(
    df_fat_avaliacoes_clientes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.gold.fat_avaliacoes_clientes")
)

print(f"✅ olist.gold.fat_avaliacoes_clientes ({df_fat_avaliacoes_clientes.count()} registros)")

In [0]:
df_ranking_produtos_aval = (
    df_avaliacoes_completo
    .groupBy("nome_produto", "categoria_produto")
    .agg(
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.count("nota_avaliacao").alias("total_avaliacoes"),
    )
    .filter(F.col("nome_produto").isNotNull())
)

df_ranking_vendedores_aval = (
    df_avaliacoes_completo
    .groupBy("id_vendedor", "estado_vendedor")
    .agg(
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.count("nota_avaliacao").alias("total_avaliacoes"),
    )
    .filter(F.col("id_vendedor").isNotNull())
)

df_produto_mais = (
    df_ranking_produtos_aval
    .orderBy(F.col("avaliacao_media").desc(), F.col("total_avaliacoes").desc())
    .limit(1)
)
print("Produto MAIS bem avaliado:")
display(df_produto_mais)

df_produto_menos = (
    df_ranking_produtos_aval
    .orderBy(F.col("avaliacao_media").asc(), F.col("total_avaliacoes").desc())
    .limit(1)
)
print("Produto MENOS bem avaliado:")
display(df_produto_menos)

df_vendedor_mais = (
    df_ranking_vendedores_aval
    .orderBy(F.col("avaliacao_media").desc(), F.col("total_avaliacoes").desc())
    .limit(1)
)
print("Vendedor MAIS bem avaliado:")
display(df_vendedor_mais)

df_vendedor_menos = (
    df_ranking_vendedores_aval
    .orderBy(F.col("avaliacao_media").asc(), F.col("total_avaliacoes").desc())
    .limit(1)
)
print("Vendedor MENOS bem avaliado:")
display(df_vendedor_menos)